<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Student Table with Team Number</title>
    <style>
        body {
            background: linear-gradient(135deg, #74ebd5, #acb6e5);
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
        }
        table {
            width: 60%;
            border-collapse: collapse;
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.2);
            border-radius: 12px;
            overflow: hidden;
            background-color: #ffffff;
        }
        th, td {
            padding: 12px 16px;
            text-align: left;
            border-bottom: 1px solid #e0e0e0;
        }
        th {
            background-color: #4a90e2;
            color: white;
            text-transform: uppercase;
            font-weight: 600;
            letter-spacing: 1px;
        }
        td {
            color: #333;
            font-size: 16px;
        }
        tr:hover {
            background-color: #f1f8ff;
            transition: background-color 0.3s ease;
        }
        tr:nth-child(even) {
            background-color: #f8f8f8;
        }
        caption {
            padding: 10px;
            font-size: 18px;
            font-weight: bold;
            color: #2c3e50;
            background-color: #dfe6e9;
            border-radius: 0 0 12px 12px;
        }
    </style>
</head>
<body>
    <table>
        <caption>Team Number: 16</caption>
        <tr>
            <th>Roll No</th>
            <th>Name</th>
            <th>USN</th>
        </tr>
        <tr>
            <td>439</td>
            <td>Spoorthi H</td>
            <td>01FE23BCS251</td>
        </tr>
        <tr>
            <td>443</td>
            <td>Ananya V H</td>
            <td>01FE23BCS104</td>
        </tr>
        <tr>
            <td>464</td>
            <td>Amogh T</td>
            <td>01FE24BCS414</td>
        </tr>
        <tr>
            <td>465</td>
            <td>Supritkumar R P</td>
            <td>01FE24BCS415</td>
        </tr>
    </table>
</body>
</html>

In [5]:

"""
Overview:
This Code implements a machine learning pipeline to predict ADHD outcomes and biological sex
using quantitative, categorical, and functional connectome data. The pipeline includes data loading,
cleaning, feature engineering, preprocessing, dimensionality reduction, ensemble modeling, and
evaluation .

Purpose:
- Predict ADHD diagnosis (ADHD_Outcome) and biological sex (Sex_F) for participants.
- Handle high-dimensional connectome data and imbalanced classes.
- Generate a submission file with predictions for a test dataset.

Pipeline Steps:
1. Data Loading: Load connectome matrices, quantitative and categorical metadata, and solutions.
2. Data Cleaning: Remove irrelevant columns and handle missing values.
3. Feature Engineering: Create interaction terms, polynomial features, and aggregates.
4. Preprocessing: Apply imputation, scaling, and encoding using a ColumnTransformer.
5. Connectome Processing: Reduce dimensionality of connectome data using PCA.
6. Feature Combination: Merge preprocessed metadata and connectome features.
7. Model Training: Train separate models for ADHD and sex using stacking and bagging ensembles.
8. Evaluation: Use cross-validation and a custom weighted F1 score to assess performance.
9. Submission: Generate predictions for the test set and save to a CSV file.

Key Features:
- Handles imbalanced data with SMOTE and balanced classifiers.
- Uses PCA to reduce connectome dimensionality.
- Employs ensemble models (HistGradientBoosting, RandomForest, MLP) for robust predictions.
- Optimizes hyperparameters via GridSearchCV.
- Custom F1 score weights female ADHD cases higher to address class imbalance.

Output:
- A CSV file ('submission.csv') with participant IDs, ADHD_Outcome, and Sex_F predictions.
"""

# Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.ensemble import BalancedRandomForestClassifier, BalancedBaggingClassifier
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.neural_network import MLPClassifier

# Set random seed for reproducibility
np.random.seed(42)

# 1. Data Loading
def load_data():
    """
    Load training and test datasets from CSV and Excel files.
    Returns: Tuple of DataFrames for connectome, quantitative, categorical data, and solutions.
    """
    train_connectome = pd.read_csv("TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
    test_connectome = pd.read_csv("TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")
    train_quantitative = pd.read_excel("TRAIN_QUANTITATIVE_METADATA_new.xlsx")
    train_categorical = pd.read_excel("TRAIN_CATEGORICAL_METADATA_new.xlsx")
    test_quantitative = pd.read_excel("TEST_QUANTITATIVE_METADATA.xlsx")
    test_categorical = pd.read_excel("TEST_CATEGORICAL.xlsx")
    train_solutions = pd.read_excel("TRAINING_SOLUTIONS.xlsx")
    return (train_connectome, test_connectome, train_quantitative, train_categorical,
            test_quantitative, test_categorical, train_solutions)

# Load datasets
(train_connectome, test_connectome, train_quant, train_cat,
 test_quant, test_cat, train_solutions) = load_data()

# 2. Data Cleaning & Feature Engineering
# Drop irrelevant columns that are unlikely to contribute to predictions
drop_cols = [
    "Basic_Demos_Study_Site", "MRI_Track_Scan_Location",
    "PreInt_Demos_Fam_Child_Ethnicity", "PreInt_Demos_Fam_Child_Race",
    'Barratt_Barratt_P1_Occ', 'Barratt_Barratt_P2_Occ',
    'MRI_Track_Scanner_Model', 'MRI_Track_Scanner_Manufacturer'
]
train_cat = train_cat.drop(columns=drop_cols, errors='ignore')
test_cat = test_cat.drop(columns=drop_cols, errors='ignore')

# Handle missing values in categorical data
for col in train_cat.columns:
    if train_cat[col].isnull().any():
        if train_cat[col].nunique() < 10:  # Low-cardinality: fill with mode
            mode_val = train_cat[col].mode()[0]
            train_cat[col].fillna(mode_val, inplace=True)
            test_cat[col].fillna(mode_val, inplace=True)
        else:  # High-cardinality: fill with 'Missing'
            train_cat[col].fillna('Missing', inplace=True)
            test_cat[col].fillna('Missing', inplace=True)

# Enhanced feature engineering
def enhanced_engineer_features(df):
    """
    Create new features and impute missing numeric values.
    Features include interaction terms, polynomial features, and aggregates.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())  # Median imputation
    # Interaction terms
    if 'EHQ_EHQ_Total' in df.columns and 'APQ_P_APQ_P_CP' in df.columns:
        df['EHQ_APQ_ratio'] = df['EHQ_EHQ_Total'] / (df['APQ_P_APQ_P_CP'] + 1e-6)
    if 'SDQ_SDQ_Total' in df.columns and 'CBCL_CBCL_Externalizing' in df.columns:
        df['SDQ_CBCL_ext_diff'] = df['SDQ_SDQ_Total'] - df['CBCL_CBCL_Externalizing']
    # Polynomial features
    if 'CBCL_CBCL_Total' in df.columns:
        df['CBCL_squared'] = df['CBCL_CBCL_Total'] ** 2
    if 'SDQ_SDQ_Total' in df.columns:
        df['SDQ_log'] = np.log1p(df['SDQ_SDQ_Total'])
    # Aggregates for SDQ-related features
    sdq_cols = [col for col in df.columns if 'SDQ_' in col]
    if len(sdq_cols) > 0:
        df['SDQ_mean'] = df[sdq_cols].mean(axis=1)
        df['SDQ_std'] = df[sdq_cols].std(axis=1)
    return df

train_quant = enhanced_engineer_features(train_quant)
test_quant = enhanced_engineer_features(test_quant)

# Merge datasets on participant_id
train_df = train_cat.merge(train_quant, on='participant_id').merge(train_solutions, on='participant_id')
test_df = test_cat.merge(test_quant, on='participant_id')

# 3. Preprocessing Pipeline
# Define feature columns
cat_cols = [col for col in train_df.select_dtypes(include=['object']).columns if col != 'participant_id']
quant_cols = [col for col in train_df.select_dtypes(include=[np.number]).columns
              if col not in ['participant_id', 'ADHD_Outcome', 'Sex_F']]

# Numeric pipeline: KNN imputation and robust scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=3)),  # Fewer neighbors for efficiency
    ('scaler', RobustScaler(quantile_range=(5, 95)))  # Robust to outliers
])

# Categorical pipeline: Impute with 'Missing' and encode with OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=0.05))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, quant_cols),
        ('cat', categorical_transformer, cat_cols)
    ], n_jobs=-1  # Parallel processing
)

# 4. Connectome Data Processing
# Extract participant IDs and numeric features
train_connectome_ids = train_connectome['participant_id']
test_connectome_ids = test_connectome['participant_id']
numeric_connectome_cols = train_connectome.select_dtypes(include=[np.number]).columns
train_connectome_features = train_connectome[numeric_connectome_cols].fillna(
    train_connectome[numeric_connectome_cols].median())
test_connectome_features = test_connectome[numeric_connectome_cols].fillna(
    test_connectome[numeric_connectome_cols].median())

# Reduce dimensionality with PCA
pca = PCA(n_components=150, random_state=42, svd_solver='arpack')  # Fixed components for efficiency
train_connectome_reduced = pca.fit_transform(train_connectome_features)
test_connectome_reduced = pca.transform(test_connectome_features)

# Create DataFrames with participant IDs as index
train_connectome_df = pd.DataFrame(train_connectome_reduced, index=train_connectome_ids)
test_connectome_df = pd.DataFrame(test_connectome_reduced, index=test_connectome_ids)

# 5. Combine Features
# Ensure consistent data types for merging
train_df['participant_id'] = train_df['participant_id'].astype(str)
train_connectome_df.index = train_connectome_df.index.astype(str)
common_ids = train_df['participant_id'].isin(train_connectome_df.index)
train_df = train_df[common_ids]
y_full_train = train_df[['ADHD_Outcome', 'Sex_F']].values

# Combine preprocessed metadata and connectome features
X_full_train = np.hstack([
    preprocessor.fit_transform(train_df[cat_cols + quant_cols]),
    train_connectome_df.loc[train_df['participant_id']].values
])

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_full_train, y_full_train, test_size=0.15, stratify=y_full_train[:, 0], random_state=42
)

# 6. Model Training & Evaluation
def weighted_f1_score(y_true, y_pred, weight_female_adhd=2.5):
    """
    Custom weighted F1 score: 70% weight on ADHD, 30% on sex, with higher weight for female ADHD cases.
    """
    y_true_adhd, y_true_sex = y_true[:, 0], y_true[:, 1]
    y_pred_adhd, y_pred_sex = y_pred[:, 0], y_pred[:, 1]
    sample_weights = np.ones_like(y_true_adhd)
    female_adhd_mask = (y_true_adhd == 1) & (y_true_sex == 1)
    sample_weights[female_adhd_mask] = weight_female_adhd
    f1_adhd = f1_score(y_true_adhd, y_pred_adhd, sample_weight=sample_weights)
    f1_sex = f1_score(y_true_sex, y_pred_sex)
    return 0.7 * f1_adhd + 0.3 * f1_sex

weighted_f1_scorer = make_scorer(weighted_f1_score, greater_is_better=True)

# ADHD Model: Stacking ensemble
adhd_base_models = [
    ('hgb', HistGradientBoostingClassifier(random_state=42, max_iter=200, learning_rate=0.1,
                                          max_depth=6, min_samples_leaf=15, early_stopping=True,
                                          l2_regularization=0.1)),
    ('rf', BalancedRandomForestClassifier(random_state=42, n_estimators=150, max_depth=8,
                                         class_weight='balanced_subsample', max_samples=0.8)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(50,), early_stopping=True, alpha=0.01,
                          random_state=42, max_iter=200))
]
adhd_model = StackingClassifier(
    estimators=adhd_base_models,
    final_estimator=LogisticRegression(class_weight='balanced', penalty='l1', solver='liblinear'),
    stack_method='auto', n_jobs=-1
)

# Sex Model: Balanced bagging ensemble
sex_model = BalancedBaggingClassifier(
    estimator=RandomForestClassifier(class_weight='balanced_subsample', max_depth=7,
                                     n_estimators=100, max_features='sqrt'),
    n_estimators=8, sampling_strategy='auto', replacement=True, random_state=42
)

# Pipelines with SMOTE and feature selection
adhd_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42, sampling_strategy=0.7, k_neighbors=3)),
    ('feature_select', SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', random_state=42),
                                      max_features=150)),
    ('model', adhd_model)
])
sex_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42, sampling_strategy=0.7, k_neighbors=3)),
    ('feature_select', RFE(estimator=LogisticRegression(solver='liblinear'),
                           n_features_to_select=80, step=15)),
    ('model', sex_model)
])

# Hyperparameter grids for optimization
param_grid_adhd = {
    'model__hgb__learning_rate': [0.1],
    'model__hgb__max_depth': [5, 6],
    'smote__k_neighbors': [3]
}
param_grid_sex = {
    'model__estimator__max_depth': [6, 7],
    'feature_select__n_features_to_select': [70, 80]
}

# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Train ADHD model with GridSearchCV
print("Training ADHD Model...")
adhd_cv = GridSearchCV(adhd_pipeline, param_grid_adhd, cv=skf.split(X_train, y_train[:, 0]),
                       scoring=make_scorer(f1_score), n_jobs=-1, verbose=1)
adhd_cv.fit(X_train, y_train[:, 0])
best_adhd_model = adhd_cv.best_estimator_
print(f"Best ADHD params: {adhd_cv.best_params_}")
print(f"Best ADHD F1: {adhd_cv.best_score_:.4f}")

# Train Sex model with GridSearchCV
print("\nTraining Sex Model...")
sex_cv = GridSearchCV(sex_pipeline, param_grid_sex, cv=skf.split(X_train, y_train[:, 1]),
                      scoring=make_scorer(f1_score), n_jobs=-1, verbose=1)
sex_cv.fit(X_train, y_train[:, 1])
best_sex_model = sex_cv.best_estimator_
print(f"Best Sex params: {sex_cv.best_params_}")
print(f"Best Sex F1: {sex_cv.best_score_:.4f}")

# Evaluate on validation set
print("\nValidation Set Performance:")
y_val_pred_adhd = best_adhd_model.predict(X_val)
y_val_pred_sex = best_sex_model.predict(X_val)
print("\nADHD_Outcome Classification Report:")
print(classification_report(y_val[:, 0], y_val_pred_adhd))
print("\nSex_F Classification Report:")
print(classification_report(y_val[:, 1], y_val_pred_sex))
weighted_f1_val = weighted_f1_score(y_val, np.column_stack((y_val_pred_adhd, y_val_pred_sex)))
print(f"\nWeighted F1 Score: {weighted_f1_val:.4f}")

# 7. Final Training & Submission
print("\nTraining final models on full data...")
final_adhd_model = best_adhd_model.fit(X_full_train, y_full_train[:, 0])
final_sex_model = best_sex_model.fit(X_full_train, y_full_train[:, 1])

# Generate test predictions
test_df['participant_id'] = test_df['participant_id'].astype(str)
test_connectome_df.index = test_connectome_df.index.astype(str)
X_test = np.hstack([
    preprocessor.transform(test_df[cat_cols + quant_cols]),
    test_connectome_df.loc[test_df['participant_id']].values
])
adhd_preds = final_adhd_model.predict(X_test)
sex_preds = final_sex_model.predict(X_test)

# Save submission file
submission = pd.DataFrame({
    'participant_id': test_df['participant_id'],
    'ADHD_Outcome': adhd_preds,
    'Sex_F': sex_preds
})
submission.to_csv('submission.csv', index=False)
print("\nEnhanced submission file saved as 'enhanced_submission.csv'")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9132\1245395927.py:92: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_cat[col].fillna(mode_val, inplace=True)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9132\1245395927.py:93: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

Training ADHD Model...
Fitting 5 folds for each of 2 candidates, totalling 10 fits


C:\Users\LENOVO\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\LENOVO\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.Crea

Best ADHD params: {'model__hgb__learning_rate': 0.1, 'model__hgb__max_depth': 5, 'smote__k_neighbors': 3}
Best ADHD F1: 0.8458

Training Sex Model...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Sex params: {'feature_select__n_features_to_select': 70, 'model__estimator__max_depth': 7}
Best Sex F1: 0.5468

Validation Set Performance:

ADHD_Outcome Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.63      0.63        57
           1       0.83      0.83      0.83       125

    accuracy                           0.77       182
   macro avg       0.73      0.73      0.73       182
weighted avg       0.77      0.77      0.77       182


Sex_F Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.70      0.77       122
           1       0.56      0.75      0.64        60

    accuracy                           0.72       182
   macro avg       0.70      0.73